<a href="https://colab.research.google.com/github/daningMontano/ACUS_ECOSYSTEM_FUNCTION_ZCH/blob/main/Notebooks/Init_biophysical_charact/1_Base_line.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Initial biophysical characterization**

By: Daning Montaño
Date: 3/5/2026

This workbook provides basic information for characterizing ACUS ecosystems. It focuses on the use of GEE and remote sensing.

# **1. Libraries**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
import geemap
import ee


# **2. Initialize**

In [3]:
# Trigger the authentication flow.
#  Autenticar la cuenta de Google Earth Engine
ee.Authenticate()
ee.Initialize(project='monitoreofpachamama')

# **3. Data**

## 2.1. AOI

In [4]:
aoi =  ee.FeatureCollection("projects/monitoreofpachamama/assets/ZamoraCH/Provincia_ZCh")
aoi_buffer_5km = ee.FeatureCollection("projects/monitoreofpachamama/assets/ZamoraCH/Provincia_ZCh_5km_buffer")

## 2.1. Remote sensing data

### DEM

30 meters

In [6]:
dem = ee.Image("USGS/SRTMGL1_003")
slope = ee.Terrain.slope(dem)

### Precipitation

5.6 km

In [5]:
chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")

### Surface temperature

1 km

In [8]:
# Cargar colección de imágenes MODIS - Temperatura superficial (LST)
# MOD11A2: 8-day composite of Land Surface Temperature
# Bandas: LST_Day_1km, LST_Night_1km, QC_Day, QC_Night

start_date = '2025-01-01'
end_date = '2025-12-31'

# Cargar MODIS LST
modis_lst = ee.ImageCollection('MODIS/061/MOD11A2') \
    .filterDate(start_date, end_date) \
    .select(['LST_Day_1km', 'LST_Night_1km', 'QC_Day', 'QC_Night'])

# Ver información de la colección
print("MODIS LST Collection:")
print(f"Cantidad de imágenes: {modis_lst.size().getInfo()}")
print(f"Primeras 5 imágenes:")
print(modis_lst.limit(5).getInfo())

MODIS LST Collection:
Cantidad de imágenes: 46
Primeras 5 imágenes:
{'type': 'ImageCollection', 'bands': [], 'version': 1777768883598403, 'id': 'MODIS/061/MOD11A2', 'properties': {'system:is_global': 1}, 'features': [{'type': 'Image', 'bands': [{'id': 'LST_Day_1km', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'dimensions': [43200, 21600], 'crs': 'SR-ORG:6974', 'crs_transform': [926.6254331383326, 0, -20015109.355797, 0, -926.6254331391667, 10007554.677903]}, {'id': 'LST_Night_1km', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'dimensions': [43200, 21600], 'crs': 'SR-ORG:6974', 'crs_transform': [926.6254331383326, 0, -20015109.355797, 0, -926.6254331391667, 10007554.677903]}, {'id': 'QC_Day', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 255}, 'dimensions': [43200, 21600], 'crs': 'SR-ORG:6974', 'crs_transform': [926.6254331383326, 0, -20015109.355797, 0, -926.6254331391667, 10007554.677903]}

Temperature 2m

11.12 km

In [9]:
# Colección ERA5-LAND HOURLY
temperature_collection = (
    ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR")
    .filterBounds(aoi)
    .select("temperature_2m")
)
print(f"ERA5-LAND MONTHLY_AGGR collection filtered for AOI: {temperature_collection.size().getInfo()} images found.")

ERA5-LAND MONTHLY_AGGR collection filtered for AOI: 914 images found.


### MODIS

500 m

In [10]:
# Load MODIS NDVI collections (v061)
modis_terra = ee.ImageCollection('MODIS/061/MOD13Q1').filterBounds(aoi)
modis_aqua = ee.ImageCollection('MODIS/061/MYD13Q1').filterBounds(aoi)

def mask_clouds_modis(img):
    # Obtener banda QA simplificada
    qa = img.select('SummaryQA')
    # En MODIS SummaryQA, 0 es 'Good Data' y 1 es 'Marginal Data'
    # Filtramos para quedarnos con datos confiables (bits 0 y 1)
    mask = qa.bitwiseAnd(3).lte(1)
    return img.updateMask(mask).select('EVI').divide(10000) \
              .copyProperties(img, ['system:time_start'])

# Aplicar máscara y combinar colecciones
filtered_terra = modis_terra.map(mask_clouds_modis)
filtered_aqua = modis_aqua.map(mask_clouds_modis)
merged_ndvi_collection = filtered_terra.merge(filtered_aqua).sort('system:time_start')

print(f"Colección NDVI procesada. Total de imágenes: {merged_ndvi_collection.size().getInfo()}")
print(f"Primer registro: {merged_ndvi_collection.first().date().format('YYYY-MM-DD').getInfo()}")

Colección NDVI procesada. Total de imágenes: 1150
Primer registro: 2000-02-49


# **4. Analysis**

## 4.1. Topographic

### DEM

In [ ]:
dem_aoi = dem.clip(aoi)

## Map de DEM

# Create a geemap.Map object
map_dem = geemap.Map()


# Add the clipped DEM layer to the map
map_dem.addLayer(dem_aoi, dem_vis_params, 'Digital Elevation Model')

# Center the map on the AOI and display it
map_dem.centerObject(aoi, 9)
map_dem
